In [1]:
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.functions import col, struct, pandas_udf
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark import SparkConf
from datasets import load_dataset
import os
import pandas as pd
from sentence_embedder import HuggingFaceSentenceEmbedder
# from synapse.ml.hf import HuggingFaceSentenceEmbedder

/home/rishic/anaconda3/envs/spark-trt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_rapids_jar():
    import requests

    SPARK_RAPIDS_VERSION = "24.10.0"
    SCALA_VERSION = "2.12"
    rapids_jar = f"rapids-4-spark_{SCALA_VERSION}-{SPARK_RAPIDS_VERSION}.jar"
    if not os.path.exists(rapids_jar):
        print("Downloading spark rapids jar")
        url = f"https://repo1.maven.org/maven2/com/nvidia/rapids-4-spark_{SCALA_VERSION}/{SPARK_RAPIDS_VERSION}/{rapids_jar}"
        response = requests.get(url)
        if response.status_code == 200:
            with open(rapids_jar, "wb") as f:
                f.write(response.content)
            print(f"File '{rapids_jar}' downloaded and saved successfully.")
        else:
            print(f"Failed to download the file. Status code: {response.status_code}")
    else:
        print("File already exists. Skipping download.")
    return rapids_jar

def initialize_spark(rapids_jar: str):
    '''
    If no active Spark session is found, initialize and configure a new one. 
    '''
    import socket
    hostname = socket.gethostname()
    conda_env = os.environ.get('CONDA_PREFIX')

    conf = SparkConf()
    conf.setMaster(f"spark://{hostname}:7077") # Assuming master is on host and default port. 
    conf.set("spark.executor.cores", "8")
    conf.set("spark.task.maxFailures", "1")
    conf.set("spark.executor.memory", "8g")
    conf.set("spark.rpc.message.maxSize", "1024")
    conf.set("spark.sql.pyspark.jvmStacktrace.enabled", "true")
    conf.set("spark.sql.execution.pyspark.udf.simplifiedTraceback.enabled", "false")
    conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    conf.set("spark.python.worker.reuse", "true")
    conf.set("spark.rapids.ml.uvm.enabled", "true")
    conf.set("spark.task.resource.gpu.amount", "1")
    conf.set("spark.executor.resource.gpu.amount", "1")
    conf.set("spark.jars", rapids_jar)
    conf.set("spark.pyspark.python", f"{conda_env}/bin/python")
    conf.set("spark.pyspark.driver.python", f"{conda_env}/bin/python")
    conf.set("spark.executorEnv.PYTHONPATH", rapids_jar)
    conf.set("spark.executorEnv.LD_LIBRARY_PATH", f"{conda_env}/lib:{conda_env}/lib/python3.11/site-packages/nvidia_pytriton.libs:$LD_LIBRARY_PATH")
    conf.set("spark.rapids.memory.gpu.minAllocFraction", "0.0001")
    conf.set("spark.plugins", "com.nvidia.spark.SQLPlugin")
    conf.set("spark.locality.wait", "0s")
    conf.set("spark.sql.cache.serializer", "com.nvidia.spark.ParquetCachedBatchSerializer")
    conf.set("spark.rapids.memory.gpu.pooling.enabled", "false")
    conf.set("spark.sql.execution.sortBeforeRepartition", "false")
    conf.set("spark.rapids.sql.format.parquet.reader.type", "MULTITHREADED")
    conf.set("spark.rapids.sql.format.parquet.multiThreadedRead.maxNumFilesParallel", "20")
    conf.set("spark.rapids.sql.multiThreadedRead.numThreads", "20")
    conf.set("spark.rapids.sql.python.gpu.enabled", "true")
    conf.set("spark.rapids.memory.pinnedPool.size", "2G")
    conf.set("spark.python.daemon.module", "rapids.daemon")
    conf.set("spark.rapids.sql.batchSizeBytes", "512m")
    conf.set("spark.sql.adaptive.enabled", "false")
    conf.set("spark.sql.files.maxPartitionBytes", "512m")
    conf.set("spark.rapids.sql.concurrentGpuTasks", "4")
    conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1000")
    conf.set("spark.rapids.sql.explain", "NONE")
    
    spark = SparkSession.builder.appName("spark-dl-embeddings").config(conf=conf).getOrCreate()
    return spark

# Check if Spark session is already active, if not, initialize it
if 'spark' not in globals():
    print("No active Spark session found, initializing manually.")
    rapids_jar = os.environ.get('RAPIDS_JAR')
    if rapids_jar is None:
        rapids_jar = get_rapids_jar()
    spark = initialize_spark(rapids_jar)
else:
    print("Using existing Spark session.")

No active Spark session found, initializing manually.
File already exists. Skipping download.


24/11/16 01:09:20 WARN Utils: Your hostname, cb4ae00-lcedt resolves to a loopback address: 127.0.1.1; using 10.110.47.100 instead (on interface eno1)
24/11/16 01:09:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
24/11/16 01:09:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/16 01:09:20 WARN ResourceUtils: The configuration of cores (exec = 8 task = 1, runnable tasks = 8) will result in wasted resources due to resource gpu limiting the number of runnable tasks per executor to: 1. Please adjust your configuration.
24/11/16 01:09:20 WARN RapidsPluginUtils: RAPIDS Accelerator 24.10.0 using cudf 24.10.0, private revision bd4e99e18e20234ee0c54f95f4b0bfce18a6255e
24/11/16 01:09:20 WARN RapidsPluginUtils: The current setting of spark.task.resource.gpu.amou

In [3]:
spark.sparkContext.addPyFile("sentence_embedder.py")

In [4]:
imdb_test = load_dataset("imdb", split="test").to_pandas().drop(columns="label")
df = spark.createDataFrame(imdb_test).limit(1024).repartition(8)

In [5]:
def preprocess(text: pd.Series) -> pd.Series:
    @pandas_udf("string")
    def _preprocess(text: pd.Series) -> pd.Series:
        return pd.Series([s.split(".")[0] for s in text])
    return _preprocess(text)

In [6]:
input_df = df.select(preprocess(col("text")).alias("input")).cache()

In [7]:
input_df.show(truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                               input|
+----------------------------------------------------------------------------------------------------+
|its a totally average film with a few semi-alright action sequences that make the plot seem a lit...|
|                                                                       This flick is a waste of time|
|       Ben, (Rupert Grint), is a deeply unhappy adolescent, the son of his unhappily married parents|
|                                                          I gave this a 3 out of a possible 10 stars|
|I quite enjoyed The Wrecking Crew (1999), which was the last of the three films in this series (t...|
|                                            I have no idea how anyone can give this movie high marks|
|                                                             OK, so my s

## Sentence Embedding

In [8]:
model_name = 'all-mpnet-base-v2'
# model_name = 'all-MiniLM-L6-v2'

embedder = HuggingFaceSentenceEmbedder(modelName=model_name,
                                       runtime="tensorrt", 
                                       batchSize=64)
embedder = embedder.setInputCol("input").setOutputCol("embeddings")

In [9]:
%%time
# first pass caches model/fn
output_df = embedder.transform(input_df)
results = output_df.collect()

CPU times: user 17.6 ms, sys: 15.7 ms, total: 33.3 ms
Wall time: 6.4 s


In [10]:
%%time
output_df = embedder.transform(input_df)
results = output_df.collect()

CPU times: user 21 ms, sys: 9.43 ms, total: 30.4 ms
Wall time: 2.63 s


In [11]:
%%time
output_df = embedder.transform(input_df)
results = output_df.collect()

CPU times: user 36.2 ms, sys: 6.22 ms, total: 42.4 ms
Wall time: 2.64 s


In [12]:
output_df.show(truncate=50)

+--------------------------------------------------+--------------------------------------------------+
|                                             input|                                        embeddings|
+--------------------------------------------------+--------------------------------------------------+
|its a totally average film with a few semi-alri...|[-0.010278629, -0.027883276, 0.01981007, 0.0183...|
|                     This flick is a waste of time|[0.006499006, 0.05473598, 0.0074297334, 0.04326...|
|Ben, (Rupert Grint), is a deeply unhappy adoles...|[-0.023290355, 0.009684379, -0.007833482, 0.001...|
|        I gave this a 3 out of a possible 10 stars|[0.031612184, -0.028978262, -0.02021819, 0.0092...|
|I quite enjoyed The Wrecking Crew (1999), which...|[-0.030412057, 0.027917933, 0.03711027, -0.0270...|
|I have no idea how anyone can give this movie h...|[-0.0025512672, 0.046100114, -0.00976657, 0.015...|
|           OK, so my summary line is a cheap trick|[0.035695426

## Sentence Embedding with Triton

In [13]:
model_name = 'all-mpnet-base-v2'

triton_embedder = HuggingFaceSentenceEmbedder(modelName=model_name,
                                       runtime="tensorrt", 
                                       batchSize=64)
triton_embedder = triton_embedder.setInputCol("input").setOutputCol("embeddings")

In [14]:
# starts triton server and loads model
triton_embedder.startTriton(num_nodes=1)

2024-11-16 01:09:40,051 - INFO - embedder: Reqesting stage-level resources: (cores=8, gpu=1.0)
2024-11-16 01:09:40,052 - INFO - embedder: Starting Triton servers on all nodes.


Triton Server PIDs:
 {
    "cb4ae00-lcedt": 1103132
}


In [15]:
%%time
output_df = embedder.transform(input_df)
results = output_df.collect()

CPU times: user 23.8 ms, sys: 5.82 ms, total: 29.7 ms
Wall time: 2.63 s


In [16]:
%%time
output_df = embedder.transform(input_df)
results = output_df.collect()

CPU times: user 25.7 ms, sys: 3.66 ms, total: 29.4 ms
Wall time: 2.61 s


In [17]:
%%time
output_df = embedder.transform(input_df)
results = output_df.collect()

CPU times: user 29.1 ms, sys: 4.62 ms, total: 33.7 ms
Wall time: 2.61 s


In [18]:
output_df.show(truncate=50)

+--------------------------------------------------+--------------------------------------------------+
|                                             input|                                        embeddings|
+--------------------------------------------------+--------------------------------------------------+
|its a totally average film with a few semi-alri...|[-0.010278629, -0.027883276, 0.01981007, 0.0183...|
|                     This flick is a waste of time|[0.006499006, 0.05473598, 0.0074297334, 0.04326...|
|Ben, (Rupert Grint), is a deeply unhappy adoles...|[-0.023290355, 0.009684379, -0.007833482, 0.001...|
|        I gave this a 3 out of a possible 10 stars|[0.031612184, -0.028978262, -0.02021819, 0.0092...|
|I quite enjoyed The Wrecking Crew (1999), which...|[-0.030412057, 0.027917933, 0.03711027, -0.0270...|
|I have no idea how anyone can give this movie h...|[-0.0025512672, 0.046100114, -0.00976657, 0.015...|
|           OK, so my summary line is a cheap trick|[0.035695426

In [ ]:
triton_embedder.stopTriton()